In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Load Data
print("Step 1: Loading data...")
df = pd.read_csv('/home/diallo/Documents/Data science projects/house prediction/filtered_data.csv')
print(f"Dataset shape: {df.shape}")

Step 1: Loading data...
Dataset shape: (650420, 10)


In [24]:
# Step 2: Data Cleaning and Preprocessing
print("Step 2: Data cleaning and preprocessing...")

# Clean price column (remove commas and convert to float)
df['Valeur fonciere'] = df['Valeur fonciere'].astype(str).str.replace(',', '').str.replace('"', '').astype(float)

# Convert date to datetime
df['Date mutation'] = pd.to_datetime(df['Date mutation'], dayfirst=True)

# Handle missing values
df = df.dropna()

# Convert department code to numeric and remove non-numeric values (like '2B', '2A' for Corsica)
df['Code departement'] = pd.to_numeric(df['Code departement'], errors='coerce')
df = df.dropna(subset=['Code departement'])
df['Code departement'] = df['Code departement'].astype(int)

# Remove outliers (prices beyond 3 standard deviations)
mean_price = df['Valeur fonciere'].mean()
std_price = df['Valeur fonciere'].std()
df = df[(df['Valeur fonciere'] >= mean_price - 3*std_price) & 
        (df['Valeur fonciere'] <= mean_price + 3*std_price)]

print(f"Data shape after cleaning: {df.shape}")

Step 2: Data cleaning and preprocessing...
Data shape after cleaning: (640290, 21)


In [44]:
# Step 3: Feature Engineering
print("Step 3: Feature engineering...")

# Land ratio
df['land_ratio'] = df['Surface terrain'] / df['Surface reelle bati']

# Total surface
df['total_surface'] = df['Surface reelle bati'] + df['Surface terrain']

# Room density
df['room_density'] = df['Nombre pieces principales'] / df['Surface reelle bati']

# Property type binary (1 for Maison, 0 for Appartement)
df['is_house'] = (df['Type local'] == 'Maison').astype(int)

# Large property flag (>150m²)
df['is_large'] = (df['Surface reelle bati'] > 150).astype(int)

# Small property flag (<60m²)
df['is_small'] = (df['Surface reelle bati'] < 60).astype(int)

# Month of sale
df['month'] = df['Date mutation'].dt.month

# Season (Winter:12-2, Spring:3-5, Summer:6-8, Fall:9-11)
def get_season(month):
    if month in [12, 1, 2]:
        return 0  # Winter
    elif month in [3, 4, 5]:
        return 1  # Spring
    elif month in [6, 7, 8]:
        return 2  # Summer
    else:
        return 3  # Fall

df['season'] = df['month'].apply(get_season)

# NEW FEATURES for better accuracy
# Price per m² (normalized size)
df['price_per_m2'] = df['Valeur fonciere'] / df['Surface reelle bati']

# Price per room
df['price_per_room'] = df['Valeur fonciere'] / df['Nombre pieces principales']

# Surface terrain to bati ratio (already have land_ratio, but let's keep it clear)
df['terrain_bati_ratio'] = df['Surface terrain'] / df['Surface reelle bati']

# Interaction: size x house type
df['size_x_house'] = df['Surface reelle bati'] * df['is_house']

# Interaction: rooms x house type
df['rooms_x_house'] = df['Nombre pieces principales'] * df['is_house']

# Department squared (non-linear effect)
df['dept_squared'] = df['Code departement'] ** 2

# Surface bati squared (non-linear effect)
df['surface_squared'] = df['Surface reelle bati'] ** 2

# Add target encoding features (will be used properly in train/test split)
commune_avg_price = df.groupby('Commune')['Valeur fonciere'].mean()
df['commune_avg_price'] = df['Commune'].map(commune_avg_price)

dept_avg_price = df.groupby('Code departement')['Valeur fonciere'].mean()
df['dept_avg_price'] = df['Code departement'].map(dept_avg_price)

# Commune price rank (relative position)
df['commune_price_rank'] = df['commune_avg_price'].rank(pct=True)

print("Feature engineering completed!")

Step 3: Feature engineering...
Feature engineering completed!


In [45]:
# Step 4: Prepare Features for Modeling
print("Step 4: Preparing features for modeling...")

# Select features for modeling (with new features)
training_features = [
    'Surface reelle bati', 'Nombre pieces principales', 'Surface terrain',
    'land_ratio', 'total_surface', 'room_density', 'is_house', 
    'is_large', 'is_small', 'month', 'season', 'Code departement',
    'commune_avg_price', 'dept_avg_price', 'commune_price_rank',
    'size_x_house', 'rooms_x_house', 'dept_squared', 'surface_squared'
]

# Target variable
target = 'Valeur fonciere'

# Prepare X and y
X = df[training_features]
y = df[target]

# Handle missing values in target-encoded features
X['commune_avg_price'] = X['commune_avg_price'].fillna(y.mean())
X['dept_avg_price'] = X['dept_avg_price'].fillna(y.mean())
X['commune_price_rank'] = X['commune_price_rank'].fillna(0.5)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Step 4: Preparing features for modeling...
Features shape: (640290, 19)
Target shape: (640290,)


In [46]:
# Step 5: Split Data into Train and Test Sets
print("Step 5: Splitting data into train and test sets...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

Step 5: Splitting data into train and test sets...
Training set size: (512232, 19)
Test set size: (128058, 19)


In [48]:
# Step 6: Train Random Forest
print("Step 6: Training Random Forest...")

from sklearn.ensemble import RandomForestRegressor

# Create pipeline with Random Forest
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_split=5, random_state=42, n_jobs=-1))
])

# Train the model
pipeline.fit(X_train, y_train)

# Predict and evaluate
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"Random Forest R² Score: {r2:.4f}")
print("Model training completed!")

Step 6: Training Random Forest...
Random Forest R² Score: 0.7017
Model training completed!


In [30]:
# Step 7: Evaluate Model Performance
print("="*50)
print("STEP 7: MODEL PERFORMANCE EVALUATION")
print("="*50)

# Make predictions on test set
y_pred = pipeline.predict(X_test)

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"\nModel Performance Metrics:")
print(f"Mean Absolute Error (MAE): €{mae:,.2f}")
print(f"Root Mean Squared Error (RMSE): €{rmse:,.2f}")
print(f"R² Score: {r2:.4f}")

STEP 7: MODEL PERFORMANCE EVALUATION

Model Performance Metrics:
Mean Absolute Error (MAE): €12,501,553.23
Root Mean Squared Error (RMSE): €24,677,289.37
R² Score: 0.3949


In [ ]:
# Cross-validation score
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
print(f"\nCross-validation R² scores: {cv_scores}")
print(f"Mean CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f}")

In [ ]:
# Plot actual vs predicted prices
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Price (€)')
plt.ylabel('Predicted Price (€)')
plt.title('Actual vs Predicted Prices')
plt.show()

In [ ]:
# Residual plot
residuals = y_test - y_pred
plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Price (€)')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.show()

In [ ]:
# Feature importance
feature_importance = pipeline.named_steps['model'].feature_importances_
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(x='importance', y='feature', data=importance_df.head(10))
plt.title('Top 10 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print(f"\nTop 10 most important features:")
print(importance_df.head(10))

In [ ]:
print("\n" + "="*50)
print("TRAINING AND EVALUATION COMPLETED!")
print("="*50)